In [2]:
# Import all related stuff and make Supabase connection
import os
from dotenv import load_dotenv, find_dotenv
from supabase import create_client
import pandas as pd

env_file = find_dotenv(usecwd=True)

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

In [14]:
# Sales table from Supabase
sales_response = supabase.table("sales").select("*").execute()
df_sales = pd.DataFrame(sales_response.data)
df_sales.head()

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [15]:
# Products table from Supabase
products_response = supabase.table("products").select("*").execute()
df_products = pd.DataFrame(products_response.data)
df_products.head()

,product_id,product_name,category,subcategory,supplier,cost_price,retail_price,eco_certified,created_at
0,1001,Praktiline seemisnahkne tennised,jalanõusid,tossud,Soome Tehdas OY,99.21,157.49,False,2022-05-12
1,1002,Mugav linane linane kostüüm,meeste_riided,ülikonnad,Riia Stils SIA,173.56,274.78,True,2022-05-10
2,1003,Elegantne orgaaniline kleit,laste_riided,kleidid,Rakvere Tekstiil OÜ,31.83,52.04,False,2020-02-19
3,1004,Minimalistlik puuvillane tuunika,naiste_riided,pluusid,Vilma Design OÜ,29.41,41.63,False,2022-12-26
4,1005,Mugav tweed kardigan,meeste_riided,kampsunid,Nordic Fashion Group OÜ,135.70,198.29,True,2021-06-19


In [16]:
# Customers table from Supabase
customers_response = supabase.table("customers").select("*").execute()
df_customers = pd.DataFrame(customers_response.data)
df_customers.head()

,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2001,Eha,Aas,eha.aas@telia.ee,+372 8713 1455,Tallinn,2024-02-27,NaN,1973
1,2002,Aivar,Kõiv,aivar.koiv@outlook.com,+372 8943 8684,Haapsalu,2025-01-09,bronze,1988
2,2003,Maris,Rebane,maris.rebane@telia.ee,+372 5918 5726,Tartu,2021-02-03,NaN,1999
3,2004,Jaak,Talvik,jaak.talvik@mail.ee,+372 8554 4232,Tallinn,2023-11-12,silver,1974
4,2005,Raivo,Koppel,raivo.koppel@yahoo.com,+372 5298 4365,Tallinn,2023-05-22,bronze,2004


In [17]:
# Merge Sales, Products and Customers tables into one df
df = (
    df_sales
    .merge(df_products, on='product_id', how='left')
    .merge(df_customers, on='customer_id', how='left')
)
df.head()



,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,...,eco_certified,created_at,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,...,False,2024-09-13,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,...,True,2021-10-18,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,...,False,2024-11-21,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,...,False,2020-07-06,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,...,True,2022-11-22,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0


In [18]:
# 1. Leia TOP 5 toodet kogukäibe järgi
top_products = df.groupby('product_name')['total_price'].sum().nlargest(5)
print(top_products)

product_name
Trendikas goretex oxfordid                    43041.80
Õhuline sünteetiline sporditossud             41237.60
Luksuslik villane pahkluu saapad              35691.84
Õhuline sünteetiline kõrge kontsaga kingad    32672.64
Praktiline viskoosne jakk                     32589.80
Name: total_price, dtype: float64


In [19]:
# 2. Leia klient suurima kogukulutusega
df['customer_name'] = df['first_name'] + ' ' + df['last_name']

top_customer_id, top_customer_name = df.groupby(['customer_id', 'customer_name'])['total_price'].sum().idxmax()
print("Top customer:", top_customer_id, top_customer_name)

Top customer: 3618.0 Tiina Pärn


In [20]:
# 1. Filtreeri: ainult 2024. aasta andmed
df['sale_date'] = pd.to_datetime(df['sale_date'])
df_2024 = df[df['sale_date'].dt.year == 2024]
print("2024 ridade arv:", len(df_2024))

2024 ridade arv: 7658


In [21]:
# 2. Sorteeri: suurimad tellimused
df.sort_values('total_price', ascending=False).head(5)

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,...,created_at,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year,customer_name
12019,12020,2407,INV-202307-00410,2023-07-08,NaN,1031,5,434.08,2170.40,online,...,2023-02-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2406,2407,2407,INV-202307-00410,2023-07-08,2073.0,1031,5,434.08,2170.40,online,...,2023-02-27,Madis,Roots,madis.roots@mail.ee,+372 5235 7953,Valga,2020-02-08,bronze,2005.0,Madis Roots
3804,3805,3805,INV-202311-00300,2023-11-22,2889.0,1069,5,402.62,2013.10,pood,...,2022-04-30,Laura,Tammik,laura.tammik@yahoo.com,+372 5110 1634,Pärnu,2022-10-16,silver,1965.0,Laura Tammik
8270,8271,8271,INV-202410-00165,2024-10-30,4053.0,1336,5,376.33,1881.65,online,...,2024-09-06,Kati,Teder,kati.teder@yahoo.com,+372 8807 4584,Tallinn,2024-04-26,NaN,1960.0,Kati Teder
1156,1157,1157,INV-202304-00274,2023-04-11,3247.0,1336,5,376.33,1881.65,pood,...,2024-09-06,Toomas,Sild,NaN,+372 8607 3580,Tartu,2021-03-21,NaN,1978.0,Toomas Sild
